# LGD Experiments: Uniform · Bimodal · Unimodal

Loads pretrained models from HuggingFace and runs LGD optimization with **seeds 0–15**.

Each cell is self-contained — run them top to bottom.
Results are **saved to disk** after the run cell, then plotted independently in separate cells.

## 1 · Setup & Imports

In [ ]:
import os, sys, gc, math, pickle, random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import torchvision.transforms.functional as TF
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from typing import List
from IPython.display import display
from huggingface_hub import login
from google.colab import userdata
import wandb
import time

# ── HuggingFace login ─────────────────────────────────────────────────────
hf_token = userdata.get('HF')
if hf_token:
    login(token=hf_token)
else:
    login()

# ── Colab: install deps + clone repo ──────────────────────────────────────
if 'google.colab' in str(get_ipython()):
    import getpass

    get_ipython().system('pip install -q diffusers transformers accelerate xformers')
    get_ipython().system('pip install -q scikit-learn matplotlib Pillow tqdm')

    github_token = userdata.get('GITHUB')
    token        = github_token if github_token else getpass.getpass('GitHub PAT: ')

    repo_url  = f'https://{token}@github.com/orineo1/conditional-matching-paper.git'
    repo_name = 'conditional-matching-paper'
    branch    = 'main'

    if not os.path.exists(repo_name):
        get_ipython().system(f'git clone {repo_url}')
    else:
        print(f"Repo '{repo_name}' already cloned — pulling latest...")
        get_ipython().system(f'cd {repo_name} && git pull')

    get_ipython().system(f'cd {repo_name} && git checkout {branch}')

    # Add MNIST src to path
    for subpath in [
        f'/content/{repo_name}',
        f'/content/{repo_name}/MNIST/src',
    ]:
        if subpath not in sys.path:
            sys.path.insert(0, subpath)

# ── WandB login ───────────────────────────────────────────────────────────
wandb_token = userdata.get('WANDB')
if wandb_token:
    wandb.login(key=wandb_token)
else:
    wandb.login()

# ── Global seed ───────────────────────────────────────────────────────────
GLOBAL_SEED = 42

def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    print(f'[Seed] All random seeds set to {seed}')

set_global_seed(GLOBAL_SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## 2 · Model Definitions

Imported directly from `MNIST/src/` in the repo.
If the path was added above these imports just work.

In [ ]:
from cond_model  import CircularAngleConsistencyModel, angles_to_circular, circular_to_angles
from uncond_model import UnconditionalUnet
from dataset      import ImprovedCNN, train_classifier

from diffusers import DDPMScheduler, DDIMScheduler
from huggingface_hub import hf_hub_download
from torch.distributions import Categorical, MultivariateNormal, MixtureSameFamily
from tqdm.notebook import tqdm

## 3 · Shared Utilities

In [ ]:
# ── MoG helpers ───────────────────────────────────────────────────────────

def mog_pdf(x, means, variances, weights=None):
    components = len(means)
    if weights is None:
        weights = torch.ones(components) / components
    pdf = torch.zeros_like(x)
    for mean, var, weight in zip(means, variances, weights):
        var_t  = torch.tensor(var,  dtype=torch.float32)
        mean_t = torch.tensor(mean, dtype=torch.float32)
        diff   = (x - mean_t + 180) % 360 - 180
        pdf   += weight * torch.exp(-0.5 * diff**2 / var_t) / (
            torch.sqrt(2 * torch.pi * var_t)
        )
    return pdf


def create_mog_pdf_evaluator(mog_means, mog_variances, weights):
    def evaluate_pdf(angles):
        return mog_pdf(
            angles,
            [m.item() for m in mog_means],
            [v.squeeze().item() for v in mog_variances],
            weights,
        )
    return evaluate_pdf


def generate_mog_samples(num_samples, means, variances, weights=None, device='cpu'):
    components = len(means)
    if weights is None:
        weights = torch.ones(components, device=device) / components
    else:
        weights = weights.to(device)
    weights    = weights / weights.sum()
    flattened  = [m.flatten() for m in means]
    dim        = flattened[0].shape[0]
    means_t    = torch.stack(flattened).to(device)
    if variances[0].numel() == dim * dim:
        covs_t = torch.stack([v.reshape(dim, dim) for v in variances]).to(device)
    else:
        covs_t = torch.stack([torch.diag(v.flatten()) for v in variances]).to(device)
    mix     = Categorical(weights)
    comp    = MultivariateNormal(means_t, covs_t)
    mixture = MixtureSameFamily(mix, comp)
    return mixture.sample((num_samples,))


def sliced_wasserstein_distance(X, Y, n_projections=50, device='cpu'):
    X    = X.to(device).float()
    Y    = Y.to(device).float()
    dim  = X.shape[1]
    proj = torch.randn(n_projections, dim, device=device)
    proj = proj / torch.norm(proj, dim=1, keepdim=True)
    X_s  = torch.sort(X @ proj.T, dim=0)[0]
    Y_s  = torch.sort(Y @ proj.T, dim=0)[0]
    return torch.mean(torch.abs(X_s - Y_s))


def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


def _build_target(mog_means, mog_variances, weights, use_uniform):
    x_range = torch.linspace(0, 360, 200)
    if use_uniform:
        target_pdf = torch.ones_like(x_range) / 360.0
    else:
        target_pdf = mog_pdf(
            x_range,
            [m.item() for m in mog_means],
            [v.squeeze().item() for v in mog_variances],
            weights,
        )
    return x_range.numpy(), target_pdf.numpy()

## 4 · Load Pretrained Models

In [ ]:
hf_token = os.environ.get('HF_TOKEN')

print('Downloading conditional model...')
cond_path = hf_hub_download(
    repo_id='Orineo/conditional-matching-paper',
    filename='MNIST/MnistConditional500Epoch.pt',
    token=hf_token,
)

print('Downloading unconditional model...')
uncond_path = hf_hub_download(
    repo_id='Orineo/conditional-matching-paper',
    filename='MNIST/MnistUncond100Epoch.pth',
    token=hf_token,
)

# Conditional
cond_model = CircularAngleConsistencyModel(
    nfeatures=2, img_features=784, eps=0.002, nunits=128, depth=5, device=device,
)
ckpt = torch.load(cond_path, map_location=device)
cond_model.load_state_dict(ckpt['model_state_dict'])
cond_model.eval()
print(f'Conditional model loaded  (epoch {ckpt["epoch"]})')

# Unconditional
uncond_model = UnconditionalUnet().to(device)
ckpt_u = torch.load(uncond_path, map_location=device)
uncond_model.load_state_dict(ckpt_u['model_state_dict'])
uncond_model.eval()
print(f'Unconditional model loaded (epoch {ckpt_u["epoch"]})')

# Noise scheduler
noise_scheduler = DDPMScheduler(num_train_timesteps=1000, beta_schedule='squaredcos_cap_v2')
print('Ready ✓')

## 5 · Train MNIST Digit Classifier

Uses `ImprovedCNN` from `MNIST/src/dataset.py` (already imported above).
Trains on standard upright MNIST — used later to classify the generated `x*` images.

**Run once** — weights saved to `best_mnist_classifier.pth`.

In [ ]:
# ---------------------------------------------------------------------------
# Classifier
# ---------------------------------------------------------------------------

NORM_MEAN       = 0.1307
NORM_STD        = 0.3081
ROTATION_ANGLES = [90, 180, 270]

import numpy as np
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from PIL import Image
from tqdm import tqdm

class ImprovedCNN(nn.Module):
    """Three-block CNN classifier for MNIST digits."""

    def __init__(self):
        super().__init__()
        self.conv1       = nn.Conv2d(1, 32, 3, padding=1)
        self.bn1         = nn.BatchNorm2d(32)
        self.conv2       = nn.Conv2d(32, 32, 3, padding=1)
        self.bn2         = nn.BatchNorm2d(32)
        self.pool1       = nn.MaxPool2d(2)
        self.dropout1    = nn.Dropout2d(0.25)

        self.conv3       = nn.Conv2d(32, 64, 3, padding=1)
        self.bn3         = nn.BatchNorm2d(64)
        self.conv4       = nn.Conv2d(64, 64, 3, padding=1)
        self.bn4         = nn.BatchNorm2d(64)
        self.pool2       = nn.MaxPool2d(2)
        self.dropout2    = nn.Dropout2d(0.25)

        self.conv5       = nn.Conv2d(64, 128, 3, padding=1)
        self.bn5         = nn.BatchNorm2d(128)
        self.pool3       = nn.MaxPool2d(2)
        self.dropout3    = nn.Dropout2d(0.25)

        self.fc1         = nn.Linear(128 * 3 * 3, 256)
        self.bn_fc1      = nn.BatchNorm1d(256)
        self.dropout_fc1 = nn.Dropout(0.5)
        self.fc2         = nn.Linear(256, 128)
        self.bn_fc2      = nn.BatchNorm1d(128)
        self.dropout_fc2 = nn.Dropout(0.5)
        self.fc3         = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool1(F.relu(self.bn2(self.conv2(F.relu(self.bn1(self.conv1(x)))))))
        x = self.dropout1(x)
        x = self.pool2(F.relu(self.bn4(self.conv4(F.relu(self.bn3(self.conv3(x)))))))
        x = self.dropout2(x)
        x = self.pool3(F.relu(self.bn5(self.conv5(x))))
        x = self.dropout3(x)
        x = x.view(-1, 128 * 3 * 3)
        x = self.dropout_fc1(F.relu(self.bn_fc1(self.fc1(x))))
        x = self.dropout_fc2(F.relu(self.bn_fc2(self.fc2(x))))
        return self.fc3(x)


def train_classifier(epochs=15, batch_size=128, lr=1e-3, device='cpu'):
    """Train ImprovedCNN on MNIST and return the best model."""
    train_transform = transforms.Compose([
        transforms.RandomRotation(30),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
        transforms.ToTensor(),
        transforms.Normalize((NORM_MEAN,), (NORM_STD,)),
    ])
    test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((NORM_MEAN,), (NORM_STD,)),
    ])

    train_loader = DataLoader(
        datasets.MNIST('./data', train=True,  download=True, transform=train_transform),
        batch_size=batch_size, shuffle=True, num_workers=2,
    )
    test_loader = DataLoader(
        datasets.MNIST('./data', train=False, download=True, transform=test_transform),
        batch_size=batch_size, shuffle=False, num_workers=2,
    )

    model     = ImprovedCNN().to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
    criterion = nn.CrossEntropyLoss()
    best_acc  = 0.0

    print(f"Training classifier — {sum(p.numel() for p in model.parameters()):,} parameters")
    for epoch in range(1, epochs + 1):
        model.train()
        correct, total = 0, 0
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            out  = model(data)
            loss = criterion(out, target)
            loss.backward()
            optimizer.step()
            correct += out.argmax(1).eq(target).sum().item()
            total   += target.size(0)

        model.eval()
        test_loss, test_correct, test_total = 0, 0, 0
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                out        = model(data)
                test_loss += criterion(out, target).item()
                test_correct += out.argmax(1).eq(target).sum().item()
                test_total   += target.size(0)

        test_acc = 100. * test_correct / test_total
        scheduler.step(test_loss / len(test_loader))
        print(f"Epoch {epoch:2d}/{epochs} | Train {100.*correct/total:.2f}% | Test {test_acc:.2f}%")

        if test_acc > best_acc:
            best_acc = test_acc
            torch.save(model.state_dict(), 'best_mnist_classifier.pth')
            print(f"  → New best: {best_acc:.2f}%")

    model.load_state_dict(torch.load('best_mnist_classifier.pth', map_location=device))
    model.eval()
    print(f"Classifier ready — best test accuracy: {best_acc:.2f}%")
    return model


In [ ]:
CLF_SAVE_PATH = 'best_mnist_classifier.pth'

if os.path.exists(CLF_SAVE_PATH):
    print(f'Classifier weights found at {CLF_SAVE_PATH} — loading...')
    digit_classifier = ImprovedCNN().to(device)
    digit_classifier.load_state_dict(torch.load(CLF_SAVE_PATH, map_location=device))
    digit_classifier.eval()
    print('Classifier loaded ✓')
else:
    print('No saved classifier found — training from scratch...')
    digit_classifier = train_classifier(epochs=15, batch_size=128, lr=1e-3, device=device)
    # train_classifier already saves to 'best_mnist_classifier.pth'
    digit_classifier.eval()
    print('Classifier trained and saved ✓')

## 6 · LGD Core

In [ ]:
def optimize_LGD(model_uncond, model_cond_cm, noise_scheduler,
                 mog_means, mog_variances, weights,
                 nsamples=500, num_x_t=10, device='cuda',
                 lr=0.01, use_uniform=False, verbose=True, seed=None,
                 num_inference_steps=300):                              # <-- add this

    if seed is not None:
        torch.manual_seed(seed)
        np.random.seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)

    ddim = DDIMScheduler.from_config(noise_scheduler.config)
    ddim.set_timesteps(num_inference_steps=num_inference_steps)        # <-- use it here
    timesteps = ddim.timesteps

    pdf_eval = (
        (lambda x: torch.ones_like(x) / 360.0)
        if use_uniform
        else create_mog_pdf_evaluator(mog_means, mog_variances, weights)
    )

    x_t = torch.randn(1, 1, 28, 28, device=device, requires_grad=True)

    iterator = tqdm(enumerate(timesteps[:-1]), total=len(timesteps)-1, leave=False) \
               if verbose else enumerate(timesteps[:-1])

    for i, t in iterator:
        x_t     = x_t.detach().clone().requires_grad_(True)
        residual      = model_uncond(x_t, torch.tensor([t], device=device))
        alpha_t       = ddim.alphas_cumprod[t]
        alpha_t_prev  = (ddim.alphas_cumprod[timesteps[i+1]]
                         if i < len(timesteps)-2 else torch.tensor(1.0))
        beta_t        = 1 - alpha_t
        pred_x0       = (x_t - beta_t**0.5 * residual) / alpha_t**0.5
        x_t_minus_1   = alpha_t_prev**0.5 * pred_x0 + (1 - alpha_t_prev)**0.5 * residual

        r_t       = torch.sqrt(beta_t)
        step_size = r_t / (1 + r_t**2) + 5 * t / 1000

        losses = []
        for _ in range(num_x_t):
            x0_sample     = pred_x0 + r_t**2 * torch.randn_like(pred_x0)
            target_angles = circular_to_angles(
                model_cond_cm.sample(nsamples=nsamples, condition_x=x0_sample,
                                     ts=[150., 50., 20., 10., 5., 1.])[0]
            )
            target_circ = angles_to_circular(target_angles)

            if use_uniform:
                mog_circ = angles_to_circular(torch.rand(nsamples, device=device) * 360)
            else:
                mog_ang  = generate_mog_samples(nsamples, mog_means, mog_variances,
                                                weights).squeeze()
                mog_circ = angles_to_circular(mog_ang)

            loss_val = sliced_wasserstein_distance(target_circ, mog_circ,
                                                   n_projections=50, device=device)
            losses.append(-loss_val)

        log_me = -torch.logsumexp(torch.stack(losses), dim=0) + math.log(num_x_t)

        if verbose:
            iterator.set_postfix(t=t.item(), loss=f'{log_me.item():.4f}')

        grad = torch.autograd.grad(log_me, x_t, retain_graph=True)[0]
        with torch.no_grad():
            # if t < 200:
            #     step_size = 0
            x_t = x_t_minus_1.detach().clone() - step_size * grad

    # Final DDIM step
    with torch.no_grad():
        last_t = timesteps[-1]
        res    = model_uncond(x_t, torch.tensor([last_t], device=device))
        a      = ddim.alphas_cumprod[last_t]
        x_t    = (x_t - (1 - a)**0.5 * res) / a**0.5

    x_final = x_t.detach().clone()

    # Final evaluation loss (4× samples)
    final_n    = nsamples * 4
    final_ang  = circular_to_angles(
        model_cond_cm.sample(nsamples=final_n, condition_x=x_final.view(1, 28, 28),
                             ts=[150., 50., 20., 10., 5., 1.])[0]
    )
    final_circ = angles_to_circular(final_ang)
    if use_uniform:
        ref_circ = angles_to_circular(torch.rand(final_n, device=device) * 360)
    else:
        ref_ang  = generate_mog_samples(final_n, mog_means, mog_variances, weights).squeeze()
        ref_circ = angles_to_circular(ref_ang)

    final_loss = sliced_wasserstein_distance(final_circ, ref_circ,
                                             n_projections=50, device=device)

    if device == 'cuda':
        torch.cuda.empty_cache()

    return x_final, final_loss

## 7 · Plot Helpers

In [ ]:
def plot_all_images(payload, ncols=5, dpi=100, save_path=None, save_no_title=False,
                    classifier=None, device=None):
    """Each row is a separate figure. 5 images per row, seed/loss/digit label."""
    results, loss_log, seed_log = payload['results'], payload['loss_log'], payload['seed_log']
    n     = len(results)
    nrows = math.ceil(n / ncols)

    if classifier is not None:
        preds = classify_generated_images(results, classifier, device, threshold=0.85)
    else:
        preds = [None] * n

    for row in range(nrows):
        start       = row * ncols
        end         = min(start + ncols, n)
        row_results = results[start:end]
        row_losses  = loss_log[start:end]
        row_seeds   = seed_log[start:end]
        row_preds   = preds[start:end]
        n_in_row    = len(row_results)

        def _make_row_fig(show_titles, _row_preds=row_preds):
            fig, axes = plt.subplots(1, ncols, figsize=(ncols * 3, 3),
                                     gridspec_kw=dict(wspace=0.02))
            axes = np.array(axes).reshape(ncols)
            for c, (img, loss, seed, pred) in enumerate(
                    zip(row_results, row_losses, row_seeds, _row_preds)):
                axes[c].imshow(img, cmap='gray')
                if show_titles:
                    title = f'Seed {seed} | Loss {loss:.4f}'
                    if classifier is not None:
                        digit_str = str(pred) if pred is not None else 'None'
                        title += f' | {digit_str}'
                    axes[c].set_title(title, fontsize=11, pad=2)
                axes[c].axis('off')
            for c in range(n_in_row, ncols):
                axes[c].axis('off')
            plt.tight_layout(pad=0.1, h_pad=0.1, w_pad=0.1)
            return fig

        fig_titled = _make_row_fig(show_titles=True)
        if save_path:
            base, ext = os.path.splitext(save_path)
            fig_titled.savefig(f'{base}_row{row+1}{ext}', dpi=dpi, bbox_inches='tight')
        plt.show()

        if save_no_title:
            fig_clean = _make_row_fig(show_titles=False)
            base, ext = os.path.splitext(save_path) if save_path else ('all_images', '.png')
            fig_clean.savefig(f'{base}_row{row+1}_notitle{ext}', dpi=dpi, bbox_inches='tight')
            plt.show()

def plot_top_k_images(payload, top_k=10, dpi=100, save_path=None, save_no_title=False):
    """Ranked grid of the top-k images by loss. Returns top_ix."""
    results, loss_log, seed_log = payload['results'], payload['loss_log'], payload['seed_log']
    experiment_name             = payload['experiment_name']
    k      = min(top_k, len(results))
    top_ix = np.argsort(loss_log)[:k]
    ncols  = min(5, k)
    nrows  = math.ceil(k / ncols)

    def _make_fig(show_titles):
        fig, axes = plt.subplots(
            nrows, ncols,
            figsize=(ncols * 3, nrows * 3),
            gridspec_kw=dict(wspace=0.02, hspace=0.1)
        )
        axes = np.array(axes).reshape(nrows, ncols)

        for rank, idx in enumerate(top_ix):
            r, c = divmod(rank, ncols)
            axes[r, c].imshow(results[idx], cmap='gray')
            if show_titles:
                axes[r, c].set_title(
                    f'Rank {rank+1} | Loss {loss_log[idx]:.4f} | Seed {seed_log[idx]}',
                    fontsize=7, pad=2
                )
            axes[r, c].axis('off')

        for rank in range(k, nrows * ncols):
            r, c = divmod(rank, ncols)
            axes[r, c].axis('off')

        if show_titles:
            plt.suptitle(f'[{experiment_name}] Top {k} Images', fontsize=12,
                         fontweight='bold', y=1.002)

        plt.tight_layout(pad=0.1, h_pad=0.1, w_pad=0.1)
        return fig

    # --- with titles ---
    fig_titled = _make_fig(show_titles=True)
    if save_path:
        fig_titled.savefig(save_path, dpi=dpi, bbox_inches='tight')
    plt.show()

    # --- without titles ---
    if save_no_title:
        fig_clean = _make_fig(show_titles=False)
        base, ext  = os.path.splitext(save_path) if save_path else (f'top{k}_clean', '.png')
        fig_clean.savefig(f'{base}_notitle{ext}', dpi=dpi, bbox_inches='tight')
        plt.show()

    return top_ix
def plot_top_k_distributions(payload, model_cond, top_ix, top_k_dist=5,
                              save_path=None, dpi=150, save_no_title=False):
    results, loss_log, seed_log = payload['results'], payload['loss_log'], payload['seed_log']
    experiment_name             = payload['experiment_name']
    x_range_np                  = payload['x_range']
    target_pdf_np               = payload['target_pdf']

    dist_k  = min(top_k_dist, len(top_ix))
    dist_ix = top_ix[:dist_k]

    # pre-sample all angles
    all_max_y = target_pdf_np.max()
    temp_angs = []
    for idx in dist_ix:
        cond_t = torch.tensor(results[idx], dtype=torch.float32).flatten().unsqueeze(0)
        ang    = circular_to_angles(
            model_cond.sample(nsamples=500, condition_x=cond_t,
                              ts=[150., 50., 20., 10., 5., 1.])[0]
        )
        temp_angs.append(ang)
        h, _ = np.histogram(ang.detach().cpu().numpy(), bins=30, range=(0, 360), density=True)
        all_max_y = max(all_max_y, h.max() if len(h) else 0)
    y_lim = all_max_y * 1.1

    target_rad_x        = np.deg2rad(x_range_np)
    target_rad_y        = target_pdf_np * (360.0 / (2 * np.pi))
    target_rad_x_closed = np.append(target_rad_x, target_rad_x[0])
    target_rad_y_closed = np.append(target_rad_y, target_rad_y[0])

    local_polar_max = target_rad_y.max()
    for ang in temp_angs:
        counts, _ = np.histogram(np.deg2rad(ang.detach().cpu().numpy()),
                                 bins=np.linspace(0, 2*np.pi, 37), density=True)
        local_polar_max = max(local_polar_max, counts.max() if len(counts) else 0)
    polar_ylim = local_polar_max * 1.1

    def _make_cart(show_titles, show_legend):
        fig, cart_axes = plt.subplots(1, dist_k, figsize=(dist_k * 4, 4), sharey=True)
        cart_axes = np.array(cart_axes).reshape(dist_k)
        for rank, (idx, ang) in enumerate(zip(dist_ix, temp_angs)):
            ang_np = ang.detach().cpu().numpy()
            ax     = cart_axes[rank]
            ax.hist(ang_np, bins=30, alpha=0.6, color='skyblue', edgecolor='black',
                    range=(0, 360), density=True, label='Sampled')
            ax.plot(x_range_np, target_pdf_np, color='orange', linewidth=2, label='Target')
            ax.fill_between(x_range_np, target_pdf_np, alpha=0.25, color='orange')
            ax.set_xlim(0, 360)
            ax.set_ylim(0, y_lim)
            ax.set_xlabel('Angle (°)', fontsize=13)
            ax.set_xticks([0, 90, 180, 270, 360])
            ax.tick_params(axis='x', labelsize=12)
            ax.tick_params(axis='y', labelsize=11)
            ax.grid(True, alpha=0.3)
            if show_titles:
                ax.set_title(f'Rank {rank+1} | Loss {loss_log[idx]:.4f} | Seed {seed_log[idx]}',
                             fontsize=9)
            if rank == 0:
                ax.set_ylabel('Density', fontsize=12)
                if show_legend:
                    ax.legend(fontsize=8)
        if show_titles:
            fig.suptitle(f'[{experiment_name}] Top {dist_k} Angle Distributions — Cartesian',
                         fontsize=13, fontweight='bold')
        fig.tight_layout()
        return fig

    def _make_polar(show_titles, show_legend):
        fig, polar_axes = plt.subplots(1, dist_k, figsize=(dist_k * 4, 4),
                                       subplot_kw={'projection': 'polar'})
        polar_axes = np.array(polar_axes).reshape(dist_k)
        for rank, (idx, ang) in enumerate(zip(dist_ix, temp_angs)):
            ang_rad     = np.deg2rad(ang.detach().cpu().numpy())
            bin_edges   = np.linspace(0, 2 * np.pi, 37)
            counts, _   = np.histogram(ang_rad, bins=bin_edges, density=True)
            bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
            width       = bin_edges[1] - bin_edges[0]
            ax = polar_axes[rank]
            ax.bar(bin_centers, counts, width=width, alpha=0.55,
                   color='skyblue', edgecolor='black', label='Sampled', zorder=2)
            ax.fill(target_rad_x_closed, target_rad_y_closed,
                    color='orange', alpha=0.35, zorder=3)
            ax.plot(target_rad_x_closed, target_rad_y_closed,
                    color='orange', linewidth=2.5, zorder=4, label='Target')
            ax.set_theta_zero_location('N')
            ax.set_theta_direction(-1)
            ax.set_yticklabels([])
            ax.set_yticks([])
            ax.tick_params(axis='x', labelsize=12)
            ax.set_ylim(0, polar_ylim)
            if show_titles:
                ax.set_title(f'Rank {rank+1} | Loss {loss_log[idx]:.4f} | Seed {seed_log[idx]}',
                             fontsize=9, pad=14)
            if rank == 0 and show_legend:
                ax.legend(fontsize=8, loc='upper right', bbox_to_anchor=(1.35, 1.15))
        if show_titles:
            fig.suptitle(f'[{experiment_name}] Top {dist_k} Angle Distributions — Polar',
                         fontsize=13, fontweight='bold')
        fig.tight_layout()
        return fig

    # --- with titles + legend ---
    fig_cart  = _make_cart (show_titles=True, show_legend=True)
    fig_polar = _make_polar(show_titles=True, show_legend=True)
    if save_path:
        base, ext = os.path.splitext(save_path)
        fig_cart .savefig(f'{base}_cart{ext}',  dpi=dpi, bbox_inches='tight')
        fig_polar.savefig(f'{base}_polar{ext}', dpi=dpi, bbox_inches='tight')
    plt.show()

    # --- without titles or legend ---
    if save_no_title:
        base, ext   = os.path.splitext(save_path) if save_path else ('dist', '.png')
        fig_cart_c  = _make_cart (show_titles=False, show_legend=False)
        fig_polar_c = _make_polar(show_titles=False, show_legend=False)
        fig_cart_c .savefig(f'{base}_cart_notitle{ext}',  dpi=dpi, bbox_inches='tight')
        fig_polar_c.savefig(f'{base}_polar_notitle{ext}', dpi=dpi, bbox_inches='tight')
        plt.show()

## 8 · Run & Save / Load Helpers

In [ ]:
def run_and_save(model_uncond, model_cond, noise_scheduler,
                 mog_means, mog_variances, weights,
                 experiment_name, seeds=range(16),
                 nsamples=500, num_x_t=10, lr=0.01, num_inference_steps=300,
                 use_uniform=False, device='cuda',
                 save_dir='results',
                 plot_each=False,          # NEW: show image + distribution after each seed
                 sample_nsamples=250):     # NEW: samples for the distribution histogram
    os.makedirs(save_dir, exist_ok=True)
    print(f'\n{"="*60}\n  EXPERIMENT: {experiment_name}\n{"="*60}')

    results, loss_log, seed_log, time_log = [], [], [], []
    for seed in seeds:
        print(f'[Seed {seed:2d}] optimizing...', end='  ')
        t0 = time.time()
        x_final, loss = optimize_LGD(
            model_uncond=model_uncond, model_cond_cm=model_cond,
            noise_scheduler=noise_scheduler,
            mog_means=mog_means, mog_variances=mog_variances, weights=weights,
            nsamples=nsamples, num_x_t=num_x_t, device=device,
            lr=lr, use_uniform=use_uniform, verbose=True, seed=seed,
            num_inference_steps=num_inference_steps
        )
        elapsed = time.time() - t0
        img      = x_final.squeeze().cpu().numpy()
        loss_val = loss.item()
        results.append(img)
        loss_log.append(loss_val)
        seed_log.append(seed)
        time_log.append(elapsed)
        print(f'loss = {loss_val:.4f}  time = {elapsed:.1f}s')

        # ── NEW: optional per-seed plot ──────────────────────────────
        if plot_each:
            _plot_single(
                img=img,
                loss_val=loss_val,
                seed=seed,
                model_cond=model_cond,
                x_final=x_final,
                mog_means=mog_means,
                mog_variances=mog_variances,
                weights=weights,
                use_uniform=use_uniform,
                sample_nsamples=sample_nsamples,
                device=device,
            )
        # ─────────────────────────────────────────────────────────────

    x_range_np, target_pdf_np = _build_target(mog_means, mog_variances, weights, use_uniform)

    payload = {
        'experiment_name': experiment_name,
        'results':         results,
        'loss_log':        loss_log,
        'seed_log':        seed_log,
        'time_log':        time_log,
        'x_range':         x_range_np,
        'target_pdf':      target_pdf_np,
        'use_uniform':     use_uniform,
    }
    save_path = os.path.join(save_dir, f'{experiment_name}.pkl')
    with open(save_path, 'wb') as f:
        pickle.dump(payload, f)

    best_i = int(np.argmin(loss_log))
    print(f'\nSaved  → {save_path}')
    print(f'Best   → seed {seed_log[best_i]}, loss {loss_log[best_i]:.4f}')
    return save_path


# ── helper: plot one image + its angle distribution ──────────────────────────
def _plot_single(img, loss_val, seed,
                 model_cond, x_final,
                 mog_means, mog_variances, weights,
                 use_uniform, sample_nsamples, device):
    """Show the generated 28×28 image and the sampled angle distribution."""
    condition_tensor = torch.tensor(img, dtype=torch.float32,
                                    device=device).flatten().unsqueeze(0)
    with torch.no_grad():
        angles = circular_to_angles(
            model_cond.sample(
                nsamples=sample_nsamples,
                condition_x=condition_tensor,
                ts=[150.0, 50.0, 20.0, 10.0, 5.0, 1.],
            )[0]
        ).detach().cpu().numpy()

    # target PDF
    x_range = torch.linspace(0, 360, 200)
    if use_uniform:
        target_pdf = torch.ones_like(x_range) / 360.0
    else:
        target_pdf = mog_pdf(
            x_range,
            [m.item() for m in mog_means],
            [v.squeeze().item() for v in mog_variances],
            weights,
        )

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    fig.suptitle(f'Seed {seed} — loss: {loss_val:.4f}', fontsize=13)

    # left: image
    axes[0].imshow(img.reshape(28, 28), cmap='gray')
    axes[0].set_title('Generated image')
    axes[0].axis('off')

    # right: angle distribution vs target
    axes[1].hist(angles, bins=30, density=True, alpha=0.6,
                 color='skyblue', edgecolor='black',
                 range=(0, 360), label='Sampled angles')
    axes[1].plot(x_range.numpy(), target_pdf.numpy(),
                 color='orange', linewidth=2, label='Target PDF')
    axes[1].set_xlim(0, 360)
    axes[1].set_xlabel('Angle (°)')
    axes[1].set_ylabel('Density')
    axes[1].set_title('Angle distribution')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    plt.close()


def load_results(save_path):
    with open(save_path, 'rb') as f:
        return pickle.load(f)

---
## 9 · Experiment 1 — Uniform Distribution

### 9a · RUN

In [ ]:
uniform_save_path = run_and_save(
    model_uncond=uncond_model, model_cond=cond_model, noise_scheduler=noise_scheduler,
    mog_means     = [torch.tensor([180], dtype=torch.float64)],
    mog_variances = [torch.tensor([[60]], dtype=torch.float64)],
    weights       = torch.tensor([1.0], dtype=torch.float64),
    experiment_name='Uniform', seeds=range(15),
    nsamples=600, num_x_t=10, num_inference_steps=300,use_uniform=True, device=device, save_dir='results',
)

### 9b · PLOT — All images (5 per row)

In [ ]:
payload_uniform = load_results(uniform_save_path)
plot_all_images(payload_uniform, ncols=5)

### 9c · PLOT — Top-k images

In [ ]:
top_ix_uniform=plot_top_k_images(payload_uniform, top_k=5, save_path='top5.png', save_no_title=True)

### 9d · PLOT — Top-k distributions (Cartesian + Polar)

In [ ]:
plot_top_k_distributions(payload_uniform, cond_model, top_ix_uniform, top_k_dist=5,save_no_title=True)

---
## 10 · Experiment 2 — Bimodal Distribution

### 10a · RUN

In [ ]:
# bimodal_means     = [torch.tensor([180], dtype=torch.float64),
#                      torch.tensor([360], dtype=torch.float64)]
# bimodal_variances = [torch.tensor([[320]], dtype=torch.float64)] * 2
# bimodal_weights   = torch.tensor([0.5, 0.5], dtype=torch.float64)

# bimodal_save_path = run_and_save(
#     model_uncond=uncond_model, model_cond=cond_model, noise_scheduler=noise_scheduler,
#     mog_means=bimodal_means, mog_variances=bimodal_variances, weights=bimodal_weights,
#     experiment_name='Bimodal', seeds=range(15),
#     nsamples=600, num_x_t=12, num_inference_steps=200, use_uniform=False, device=device, save_dir='results',
# )
# payload_bimodal = load_results(bimodal_save_path)
# plot_all_images(payload_bimodal, ncols=5)

In [ ]:
bimodal_means     = [torch.tensor([180], dtype=torch.float64),
                     torch.tensor([360], dtype=torch.float64)]
bimodal_variances = [torch.tensor([[245]], dtype=torch.float64)] * 2
bimodal_weights   = torch.tensor([0.5, 0.5], dtype=torch.float64)

bimodal_save_path = run_and_save(
    model_uncond=uncond_model, model_cond=cond_model, noise_scheduler=noise_scheduler,
    mog_means=bimodal_means, mog_variances=bimodal_variances, weights=bimodal_weights,
    experiment_name='Bimodal', seeds=range(15),
    nsamples=600, num_x_t=10, num_inference_steps=100, use_uniform=False, device=device, save_dir='results',
)

### 10b · PLOT — All images

In [ ]:
payload_bimodal = load_results(bimodal_save_path)
plot_all_images(payload_bimodal, ncols=5)

### 10c · PLOT — Top-k images

In [ ]:
top_ix_bimodal=plot_top_k_images(payload_bimodal, top_k=5, save_path='top5.png', save_no_title=True)

### 10d · PLOT — Top-k distributions

In [ ]:
plot_top_k_distributions(payload_bimodal, cond_model, top_ix_bimodal, top_k_dist=5,save_no_title=True)

---
## 11 · Experiment 3 — Unimodal Distribution

### 11a · RUN

In [ ]:
unimodal_means     = [torch.tensor([360], dtype=torch.float64)]
unimodal_variances = [torch.tensor([[620]],  dtype=torch.float64)]
unimodal_weights   = torch.tensor([1.0],    dtype=torch.float64)

unimodal_save_path = run_and_save(
    model_uncond=uncond_model, model_cond=cond_model, noise_scheduler=noise_scheduler,
    mog_means=unimodal_means, mog_variances=unimodal_variances, weights=unimodal_weights,
    experiment_name='Unimodal', seeds=range(15),
    nsamples=600, num_x_t=10,num_inference_steps=100, use_uniform=False, device=device, save_dir='results',
)


### 11b · PLOT — All images

In [ ]:
payload_unimodal = load_results(unimodal_save_path)
plot_all_images(payload_unimodal, ncols=5)

### 11c · PLOT — Top-k images

In [ ]:
top_ix_unimodal=plot_top_k_images(payload_unimodal, top_k=5, save_path='top5.png', save_no_title=True)

### 11d · PLOT — Top-k distributions

In [ ]:
plot_top_k_distributions(payload_unimodal, cond_model, top_ix_unimodal, top_k_dist=5,save_no_title=True)

---
## 12 · Quantitative Results Table

For each experiment (row) we compute:
- **SWD-all**: mean ± std of the final SWD loss over all 15 seeds
- **SWD-top5**: mean ± std over the 5 lowest-loss seeds
- **Digit distribution**: % of the 15 generated images classified as each digit 0–9

The classifier runs on the raw numpy image arrays stored in each payload.
Images are in normalised pixel space (mean 0.1307, std 0.3081) and already 28×28.

In [ ]:
NORM_MEAN = 0.1307
NORM_STD  = 0.3081

def classify_generated_images(images_np, classifier, device, threshold=0.95):
    """
    Classify a list of 28×28 numpy arrays (already in normalised space).
    Returns a list of predicted digit labels (int) or None if below threshold.
    """
    classifier.eval()
    preds = []
    with torch.no_grad():
        for img in images_np:
            tensor = torch.tensor(img, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
            probs  = torch.softmax(classifier(tensor), dim=1)
            conf, pred = probs.max(dim=1)
            preds.append(pred.item() if conf.item() > threshold else None)
    return preds


def compute_experiment_stats(payload, classifier, device, top_k=5, threshold=0.85):
    losses = np.array(payload['loss_log'])
    times  = np.array(payload.get('time_log', [np.nan] * len(losses)))
    n      = len(losses)
    top_ix = np.argsort(losses)[:top_k]

    preds = classify_generated_images(payload['results'], classifier, device,
                                      threshold=threshold)

    digit_counts   = {d: 0 for d in range(10)}
    n_unclassified = 0
    for p in preds:
        if p is None:
            n_unclassified += 1
        else:
            digit_counts[p] += 1

    n_classified = n - n_unclassified

    return {
        'swd_all_mean':     losses.mean(),
        'swd_all_std':      losses.std(),
        'swd_top_mean':     losses[top_ix].mean(),
        'swd_top_std':      losses[top_ix].std(),
        'time_mean':        times.mean(),
        'time_std':         times.std(),
        'digit_counts':     digit_counts,
        'digit_pct':        {d: 100.0 * digit_counts[d] / n for d in range(10)},
        'n_seeds':          n,
        'n_classified':     n_classified,
        'n_unclassified':   n_unclassified,
        'pct_unclassified': 100.0 * n_unclassified / n,
        'predicted_labels': preds,
    }

### 12a · Compute stats for all three experiments

In [ ]:
TOP_K_TABLE = 5   # number of "top" seeds for SWD-top column

experiments = [
    ('Uniform',  payload_uniform),
    ('Bimodal',  payload_bimodal),
    ('Unimodal', payload_unimodal),
]

stats = {}
for name, payload in experiments:
    print(f'Classifying {name}...')
    stats[name] = compute_experiment_stats(payload, digit_classifier, device, top_k=TOP_K_TABLE)
    labels = stats[name]['predicted_labels']
    print(f'  SWD all: {stats[name]["swd_all_mean"]:.4f} ± {stats[name]["swd_all_std"]:.4f}')
    print(f'  SWD top-{TOP_K_TABLE}: {stats[name]["swd_top_mean"]:.4f} ± {stats[name]["swd_top_std"]:.4f}')
    print(f'  Digits: {labels}')

In [ ]:
import pandas as pd

def render_results_table(stats, top_k=5):
    digits = list(range(10))
    rows = []
    for exp_name, s in stats.items():
        row = {
            'Experiment':      exp_name,
            'SWD-all':         f"{s['swd_all_mean']:.4f} ± {s['swd_all_std']:.4f}",
            f'SWD-top{top_k}': f"{s['swd_top_mean']:.4f} ± {s['swd_top_std']:.4f}",
            'Time (s)':        f"{s['time_mean']:.1f} ± {s['time_std']:.1f}",
            'N seeds':         s['n_seeds'],
        }
        for d in digits:
            row[str(d)] = f"{s['digit_pct'][d]:.1f}%"
        row['None'] = f"{s.get('pct_unclassified', 0.0):.1f}%"
        rows.append(row)

    df = pd.DataFrame(rows).set_index('Experiment')
    # print(df.to_string())
    return df

df_results = render_results_table(stats, top_k=TOP_K_TABLE)
df_results

In [ ]:
plot_all_images(payload_uniform,  ncols=5, classifier=digit_classifier, device=device)
plot_all_images(payload_bimodal,  ncols=5, classifier=digit_classifier, device=device)
plot_all_images(payload_unimodal, ncols=5, classifier=digit_classifier, device=device)

In [ ]:
STEPS_TO_TRY = [235, 240, 245, 250]
N_SEEDS       = 15
GS_SAVE_DIR   = 'results/grid_search'
os.makedirs(GS_SAVE_DIR, exist_ok=True)

gs_experiments = {
    'Bimodal': {
        'mog_means':     bimodal_means,
        'mog_variances': bimodal_variances,
        'weights':       bimodal_weights,
        'use_uniform':   False,
    },
}

gs_results = {}
gs_images  = {}

THRESHOLD = 0.85
TOP_K_GS  = 5

for exp_name, cfg in gs_experiments.items():
    gs_results[exp_name] = {}
    gs_images[exp_name]  = {}
    print(f'\n{"="*60}\n  Grid search: {exp_name}\n{"="*60}')

    for n_steps in STEPS_TO_TRY:
        losses, times, images = [], [], []
        print(f'  steps={n_steps}', end='  ')

        for seed in range(N_SEEDS):
            t0 = time.time()
            img, loss = optimize_LGD(
                model_uncond=uncond_model,
                model_cond_cm=cond_model,
                noise_scheduler=noise_scheduler,
                mog_means=cfg['mog_means'],
                mog_variances=cfg['mog_variances'],
                weights=cfg['weights'],
                nsamples=600,
                num_x_t=10,
                device=device,
                use_uniform=cfg['use_uniform'],
                verbose=False,
                seed=seed,
                num_inference_steps=n_steps,
            )
            elapsed = time.time() - t0
            losses.append(loss.item())
            times.append(elapsed)
            images.append(np.squeeze(img.cpu().numpy() if hasattr(img, 'cpu') else img))
            print('.', end='', flush=True)

        gs_images[exp_name][n_steps]  = images
        gs_results[exp_name][n_steps] = {
            'mean':      np.mean(losses),
            'std':       np.std(losses),
            'time_mean': np.mean(times),
            'time_std':  np.std(times),
            'losses':    losses,
            'times':     times,
        }
        print(f'  loss={np.mean(losses):.4f}±{np.std(losses):.4f}  '
              f'time={np.mean(times):.1f}s')

        payload = {
            'results':         images,
            'loss_log':        losses,
            'seed_log':        list(range(len(losses))),
            'experiment_name': f'{exp_name} | steps={n_steps}',
        }

        print(f'\n── {exp_name} | steps={n_steps} ── Top-{TOP_K_GS} images')
        top_ix = plot_top_k_images(
            payload, top_k=TOP_K_GS,
            save_path=f'{GS_SAVE_DIR}/top{TOP_K_GS}_{exp_name}_steps{n_steps}.png',
            save_no_title=True,
        )

        print(f'\n── {exp_name} | steps={n_steps} ── All images (no classifier)')
        plot_all_images(payload, ncols=5)

        print(f'\n── {exp_name} | steps={n_steps} ── All images (with classifier)')
        plot_all_images(payload, ncols=5, classifier=digit_classifier, device=device)

        preds        = classify_generated_images(images, digit_classifier, device, threshold=THRESHOLD)
        digits       = list(range(10))
        digit_counts = {d: 0 for d in digits}
        n_unclassified = 0
        for p in preds:
            if p is None: n_unclassified += 1
            else:         digit_counts[p] += 1

        n          = len(images)
        losses_arr = np.array(losses)
        top_ix_arr = np.argsort(losses_arr)[:TOP_K_GS]

        row = {
            'Experiment': exp_name,
            'Steps':      n_steps,
            'SWD mean':   f"{np.mean(losses):.4f}",
            'SWD std':    f"{np.std(losses):.4f}",
            f'SWD top-{TOP_K_GS}': f"{losses_arr[top_ix_arr].mean():.4f}",
        }
        for d in digits:
            row[str(d)] = f"{100.0 * digit_counts[d] / n:.0f}%"
        row['None'] = f"{100.0 * n_unclassified / n:.0f}%"

        display_cols = ['Experiment', 'Steps', 'SWD mean', 'SWD std', f'SWD top-{TOP_K_GS}'] \
                     + [str(d) for d in digits] + ['None']
        df_step = pd.DataFrame([row])[display_cols].set_index(['Experiment', 'Steps'])
        print(f'\n── {exp_name} | steps={n_steps} ── Classifier table')
        display(df_step)

# ── Save ──────────────────────────────────────────────────────────────────
with open(f'{GS_SAVE_DIR}/grid_search_results_bimodal_fine.pkl', 'wb') as f:
    pickle.dump({'gs_results': gs_results, 'gs_images': gs_images}, f)
print('\nGrid search results saved.')

# ── SWD summary table ─────────────────────────────────────────────────────
rows = []
for exp_name, step_dict in gs_results.items():
    for n_steps, v in step_dict.items():
        rows.append({
            'Experiment':   exp_name,
            'Steps':        n_steps,
            'SWD mean':     f"{v['mean']:.4f}",
            'SWD std':      f"{v['std']:.4f}",
            'Time mean(s)': f"{v['time_mean']:.1f}",
            'Time std(s)':  f"{v['time_std']:.1f}",
        })
df_gs = pd.DataFrame(rows)
print(df_gs.to_string(index=False))

In [ ]:
from google.colab import runtime
runtime.unassign()
